# 🤖 VERY Simple ML: Titanic Survival Prediction

**The real-world problem**: Should the Titanic have had more lifeboats? Who gets rescued first? Let's train a model to predict survival based on passenger data.

**Don't worry — just click Runtime → Run all and watch it work.

## 1. Download the dataset

In [ ]:
# Download Titanic data (this is real data from the actual ship)
!wget -q "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv" -O titanic.csv
print("✅ Downloaded")

## 2. Load & take a peek

In [ ]:
import pandas as pd

# Load the CSV file into a table
df = pd.read_csv("titanic.csv")

# Show first 5 rows so you see what the data looks like
df.head()

**Each row = one passenger. We want to predict the 'Survived' column (1 = yes, 0 = no).**

- Pclass = Ticket class (1 = rich, 2 = middle, 3 = poor)
- Sex = male / female
- Age = age in years
- SibSp = how many siblings/spouses they brought
- Parch = how many parents/children they brought
- Fare = ticket price
- Embarked = which port they boarded at

## 3. Pick only the useful columns & clean up

In [ ]:
# Keep only the columns that might help predict survival
# Drop PassengerId, Name, Ticket, Cabin (useless for prediction)
df = df[['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']]

# Fill missing ages with the average age
df['Age'] = df['Age'].fillna(df['Age'].median())

# Fill missing embark port with the most common port
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Convert 'male'/'female' to numbers (0 and 1)
# ML models only understand numbers, not words
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

# Convert port letters to numbers
df['Embarked'] = df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

print("✅ Data is clean and ready")
df.head()

## 4. Separate: Features (X) vs Target (y)

**X** = the inputs (age, sex, class, etc.) → what the model reads

**y** = the answer (Survived or not) → what the model tries to predict

In [ ]:
# X = everything except Survived
X = df.drop(columns=['Survived'])

# y = only the Survived column
y = df['Survived']

print("Inputs (X):", list(X.columns))
print("Target (y): Survived (0 = died, 1 = survived)")

## 5. Split: Train vs Test

We **hide** 20% of the data from the model. The model learns on 80%, then we test it on the 20% it never saw.

**Why?** If we tested on data the model already saw, it'd cheat — like giving a student the exam answers before the test.

If the model is accurate on the hidden 20%, it means it learned **real patterns**, not just memorized.

In [ ]:
from sklearn.model_selection import train_test_split

# Split: 80% train, 20% test. random_state=42 = same split every time
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training examples: {len(X_train)}")
print(f"Testing examples:  {len(X_test)}")

## 6. Train the model ⚡

This is where the magic happens. The model looks at 700+ passengers and learns patterns like:
- "If female → more likely to survive"
- "If poor (3rd class) → less likely to survive"
- "If old → less likely to survive"

It figures all this out **automatically**. We don't write any rules.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Create the model (100 decision trees working together)
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Train it! This is where it learns the patterns
model.fit(X_train, y_train)

print("✅ Model trained! It now knows what patterns lead to survival.")

## 7. Test the model 👀

Now we ask the model to predict survival on the 20% of passengers it NEVER saw during training. Then we compare its predictions to the real answers.

In [ ]:
# Ask the model to predict on the hidden test data
predictions = model.predict(X_test)

# Compare predictions vs actual answers side by side
results = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': predictions
})

print("First 20 predictions vs reality:")
print(results.head(20))
print("\n✅ 1 = Survived, 0 = Died. Look at how many match!")

## 8. How accurate is it? 📊

In [ ]:
from sklearn.metrics import accuracy_score

# Accuracy = (correct predictions) / (total predictions)
accuracy = accuracy_score(y_test, predictions)

correct = (predictions == y_test).sum()
total = len(y_test)

print(f"✅ Correct: {correct} out of {total}")
print(f"🎯 Accuracy: {accuracy:.1%}")
print(f"\n🔑 If accuracy > 70%, the model learned REAL patterns.")
print(f"   (Random guessing would get ~50%)")

## 9. Where did it get confused? Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as snn
import matplotlib.pyplot as plt

# Build the confusion matrix
cm = confusion_matrix(y_test, predictions)

# Plot it as a colored grid
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Died', 'Predicted Survived'],
            yticklabels=['Actually Died', 'Actually Survived'])
plt.title('Confusion Matrix')
plt.show()

print("📖 Reading this:")
print(f"  Top-left:    {cm[0][0]} — Correctly said 'Died' ✅")
print(f"  Top-right:   {cm[0][1]} — Said 'Survived' but actually Died ❌")
print(f"  Bottom-left: {cm[1][0]} — Said 'Died' but actually Survived ❌")
print(f"  Bottom-right:{cm[1][1]} — Correctly said 'Survived' ✅")

## 10. What did the model learn? (Feature Importance)

In [ ]:
# Show which columns mattered most for prediction
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

print("What drives survival? (bigger = more important)")
print("-" * 30)
for i, row in importance.iterrows():
    bar = '█' * int(row['Importance'] * 50)
    print(f"{row['Feature']:10s} {bar} {row['Importance']:.1%}")

## 11. Predict on YOUR OWN passenger 🧑

This is the REAL use of ML — predict something we've never seen before. Change the values below and run the cell.

In [ ]:
# === CHANGE THESE VALUES TO TEST DIFFERENT PEOPLE ===
# Make your own passenger:
my_passenger = pd.DataFrame([{
    'Pclass': 1,      # 1 = rich, 2 = middle, 3 = poor
    'Sex': 1,          # 0 = male, 1 = female
    'Age': 25,         # age in years
    'SibSp': 0,        # siblings/spouses aboard
    'Parch': 0,        # parents/children aboard
    'Fare': 100,       # ticket price
    'Embarked': 0      # 0 = Southampton, 1 = Cherbourg, 2 = Queenstown
}])

# Predict
pred = model.predict(my_passenger)[0]
prob = model.predict_proba(my_passenger)[0][1]

print("=" * 45)
print("YOUR PASSENGER")
print("=" * 45)
print(f"Pclass: {my_passenger['Pclass'].values[0]} | Sex: {'Female' if my_passenger['Sex'].values[0] else 'Male'} | Age: {my_passenger['Age'].values[0]}")
print(f"\nPrediction: {'✅ SURVIVED' if pred == 1 else '❌ DIED'}")
print(f"Survival probability: {prob:.0%}")
print("\nTry changing the values above and re-run this cell!")

## 🎉 Done! You trained a real ML model on real data.

| You did this | What it means |
|---|---|
| Loaded real Titanic data | 891 actual passengers |
| Cleaned the data | Fixed missing ages, converted text to numbers |
| Trained a model | Random Forest learned survival patterns automatically |
| Tested on hidden data | ~80% accuracy on passengers it NEVER saw |
| Made a prediction | You can type in any person and predict survival |

**What to try next**:
- Change the passenger values in step 11 and re-run it
- Go back to step 6 and change `n_estimators=100` to `200` — does accuracy improve?
- Ask me: "How do I save this model and use it in a website?"